# RADBIO_NEURO_003 — Neural Simulation Layer

This notebook converts RADBIO_NEURO_002 biology states into a neural simulation layer.

**Scientific status:** this is a neural simulation scaffold. It is not yet a validated human or astronaut-health prediction.

## Model transition

Previous toy layer:

`dose_rate → ROS → mitochondrial damage → risk proxy`

Current RADBIO_NEURO_003 layer:

`dose_rate / cumulative dose → acute ROS/channel term + chronic mitochondrial/ATP term → conductance-aware LIF network → firing rate, raster, voltage traces`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.neural_layer_model import NeuralSimConfig, run_lif_network, summarize_result

BASE = Path('.')
DATA = BASE / 'data'
OUT = BASE / 'outputs'
FIG = BASE / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

selected = pd.read_csv(DATA / 'selected_shielding_biology_summary.csv')
daily = pd.read_csv(DATA / 'biology_calibrated_daily_timecourses_all_scenarios.csv')
selected.head()

## Neural parameter mapping

The fast ROS term affects excitatory/channel-like drive. The slow mitochondrial term affects ATP-dependent drive, effective leak/integration, and threshold.

In [ ]:
import inspect
import src.neural_layer_model as nlm
print(inspect.getsource(nlm.state_to_neural_parameters))

## Run endpoint simulations

The endpoint simulation uses the day-180 biology state for selected shielding values.

In [ ]:
endpoint_rows = []
for _, row in selected.iterrows():
    cfg = NeuralSimConfig(n_neurons=80, sim_ms=2000.0, dt_ms=0.1, seed=int(1000 + row['shield_mm_Al'] * 10 + (0 if row['scenario']=='LEO_ISS_like' else 500)))
    result = run_lif_network(row, cfg)
    endpoint_rows.append(summarize_result(row, result))

endpoint = pd.DataFrame(endpoint_rows)
endpoint.to_csv(OUT / 'neural_endpoint_simulation_summary.csv', index=False)
endpoint[['scenario','shield_mm_Al','dose_rate_mGy_day','atp_proxy','mean_firing_rate_hz','calibration_domain_flag']]

In [ ]:
plt.figure(figsize=(8,5))
for scenario, group in endpoint.groupby('scenario'):
    g = group.sort_values('shield_mm_Al')
    plt.plot(g['shield_mm_Al'], g['mean_firing_rate_hz'], marker='o', label=scenario)
plt.xscale('log')
plt.xlabel('Al shielding thickness (mm)')
plt.ylabel('Mean firing rate (Hz)')
plt.title('Endpoint neural firing rate after 180 days')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'endpoint_firing_rate_vs_shielding.png', dpi=200)
plt.show()

## Timecourse simulation at 10 mm Al

This samples the RADBIO_NEURO_002 daily biology timecourse every 15 days and runs a short neural simulation at each sampled state.

In [ ]:
time_rows = []
for scenario in ['LEO_ISS_like', 'VAB_RBSP_like']:
    sub = daily[(daily['scenario'] == scenario) & (daily['shield_mm_Al'] == 10.0)].copy()
    for day in sorted(set(list(range(0, 181, 15)) + [180])):
        row = sub.iloc[(sub['day'] - day).abs().argmin()].copy()
        row['ros_norm_day_end'] = row['ros_norm']
        row['mito_integrity_day_end'] = row['mito_integrity']
        row['atp_proxy_day_end'] = row['atp_proxy']
        row['fast_excitability_delta_day_end'] = row['fast_excitability_delta']
        row['slow_mito_atp_suppression_day_end'] = row['slow_mito_atp_suppression']
        row['calibration_domain_flag'] = 'timecourse_from_biology_layer'
        row['model_status'] = 'neural_scaffold_timecourse'
        cfg = NeuralSimConfig(n_neurons=60, sim_ms=1000.0, dt_ms=0.1, seed=int(3000 + day + (0 if scenario=='LEO_ISS_like' else 500)))
        result = run_lif_network(row, cfg)
        time_rows.append(summarize_result(row, result))

timecourse = pd.DataFrame(time_rows)
timecourse.to_csv(OUT / 'neural_timecourse_10mm_summary.csv', index=False)
timecourse.head()

In [ ]:
plt.figure(figsize=(8,5))
for scenario, group in timecourse.groupby('scenario'):
    g = group.sort_values('day')
    plt.plot(g['day'], g['mean_firing_rate_hz'], marker='o', label=scenario)
plt.xlabel('Mission day')
plt.ylabel('Mean firing rate (Hz)')
plt.title('Neural simulation timecourse at 10 mm Al')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'timecourse_firing_rate_10mm.png', dpi=200)
plt.show()

## Interpretation boundary

Van Allen/RBSP-like dose rates in this dataset are outside the current primary biology validation domain. Treat those runs as stress tests of the computational pipeline, not validated biological forecasts.

In [ ]:
flags = endpoint.groupby(['scenario','calibration_domain_flag']).size().reset_index(name='n_cases')
flags

## Next upgrade path

The next iteration should replace this compact LIF scaffold with a Brian2 or NEURON implementation using explicit Na/K/Ca/leak conductances and ATP-sensitive pump/KATP terms.